# 실전 데이터 분석 59: 국내선 항공사별 평균 관제 지연 시간 대조 및 출발 시간대별 기상 악화 지표 다차원 연계 분석

> **📥 [실습 주피터 노트북(.ipynb) 다운로드](practice.ipynb)**

## 📌 강의 개요 (30분 완성)

![코믹 일러스트](img/intro_comic.png)
국내선 공항의 항공편 운항 관제 로그 데이터셋입니다. 각 항공사의 정시성 효율 격차를 평균 지연 막대 그래프로 한 눈에 비교하고, 운항 예정 시각(Scheduled Time)의 흐름과 기상 악화(Weather Severity) 강도가 최종 연계 지연 분수(Delay Minutes)에 미치는 누적 효과를 분석합니다.

**학습 목표:**
* **정시 운항 처리 (`fillna`):** 지연 시간이 결측된(NaN) 데이터를 정시 운항(0분 지연) 건으로 처리하여 통계 표본의 왜곡을 방지합니다.
* **기상 강도 그라데이션 산점도 (`scatterplot`):** 하루 예정 시각(X축)과 지연 분(Y축) 위에 기상 악화도를 연속 그라데이션 색상(`hue='Weather_Severity'`)으로 얹어 관찰합니다.

---

## Step 1: 데이터 구조 살펴보기 (Data Overview)

![Step 1 데이터 구조 개념도](img/step1_dataset.svg)

`csv_data` 폴더에 준비해 둔 `flight_delays.csv` 파일을 판다스로 불러옵니다.


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 그래프 설정 (한글 폰트 및 마이너스 기호 깨짐 방지)
import koreanize_matplotlib
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style="whitegrid")

# 로컬 CSV 파일 불러오기
df = pd.read_csv('./flight_delays.csv')

# 데이터 구조 및 첫 5행 확인
print(df.info())
print(df.head())


<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   FlightID            1000 non-null   object 
 1   Airline             1000 non-null   object 
 2   ScheduledTime_Hour  1000 non-null   int64  
 3   Delay_Minutes       987 non-null    float64
 4   Weather_Severity    1000 non-null   float64
 5   IsWeekend           1000 non-null   object 
dtypes: float64(2), int64(1), object(3)
memory usage: 47.0 KB
None
  FlightID     Airline  ScheduledTime_Hour  Delay_Minutes  Weather_Severity IsWeekend
0  FL00001        Tway                  14           46.0               2.5        No
1  FL00002  Korean Air                  11           27.0               1.4       Yes
2  FL00003     Jin Air                   8           78.0               4.1        No
3  FL00004  Korean Air                   9           58.0               3.6        No
4  FL00005      Asiana                  19           77.0               4.2        No
> ```

### 💡 코드 딥다이브 (Code Deep Dive)
**주요 분석 대상 컬럼:**
* `FlightID`: 국내선 노선 개별 비행 고유 번호
* `Airline`: 취항 항공사 브랜드 (Korean Air, Asiana, Jeju Air 등)
* `ScheduledTime_Hour`: 공항 게이트 출발 예정 시각 (0~23시)
* `Delay_Minutes`: 정시 대비 실제 이륙 출발이 늦어진 지연 분수 (결측치 존재)
* `Weather_Severity`: 관찰 시점의 강풍/폭우 등 기상 악화 평가 심각 지수 (1.0 = 맑음, 5.0 = 폭풍우)
* `IsWeekend`: 주말(토/일) 출발 여부 (Yes/No)

---

## Step 2: 전처리와 결측치 정제 (Preprocess)

![Step 2 데이터 정제 개념도](img/step2_missing_values.svg)

현실의 데이터는 항상 누락이 있거나 유효성 정제가 필요합니다. 데이터 전처리 단계에서 결측 상태를 확인하고 올바르게 보정합니다.


In [ ]:
# 1. 지연 기록 결측치 개수 파악
print("--- 정제 전 지연 결측 개수 ---")
print(df['Delay_Minutes'].isnull().sum())

# 2. 결측 지연 기록은 지연이 발생하지 않은 '정시 출발(0분)'로 대치 처리
df['Delay_Minutes'] = df['Delay_Minutes'].fillna(0)

print("\n--- 정제 후 지연 결측 개수 ---")
print(df['Delay_Minutes'].isnull().sum())


--- 정제 전 지연 결측 개수 ---
13

--- 정제 후 지연 결측 개수 ---
0
> ```

### 💡 분석가의 통찰 (Analyst's Insight)
* **관제 로그의 0값 결측 처리의 타당성:** 공항 관제 시스템에서 13개 비행편의 지연 기록이 누락(NaN)된 것은 정상적인 누락이 아닌, **'지연이 0분이라 지연 경보 테이블에 아예 로그가 적재되지 않은 정시 출발'**을 의미합니다. 이를 평균 지연값(예: 30분 지연)으로 치환하면 대단히 유능했던 정시 이륙 데이터가 강제 훼손되므로, 물리적인 참값인 `0`으로 대치하여 데이터 품질을 바로잡아야 합니다.

---

## Step 3: 단변수 분포 분석 (Univariate EDA)

![Step 3 시각화 개념도](img/step3_visualization.svg)

가장 먼저 핵심 변수가 전체 데이터에서 어떤 빈도와 분포를 가졌는지 단일 변수 시각화를 통해 파악해 봅니다.


In [ ]:
plt.figure(figsize=(8, 5))

# 항공사별 지연 시간 평균 비교 막대 차트
sns.barplot(data=df, x='Airline', y='Delay_Minutes', palette='pastel', errorbar=None)

plt.title('항공사별 평균 지연 시간(Delay Minutes)', fontsize=14, fontweight='bold')
plt.xlabel('항공사 (Airline)')
plt.ylabel('평균 지연 시간 (분)')
plt.show()


> **💻 [실행 결과 시각화]**
> ![실행 결과 시각화](img/exec_step_3.svg)

### 💡 시각화 차트 읽는 법 & 인사이트
* **정시 운항 관제 능력의 시각적 대조:** 대형 풀서비스 캐리어(FSC)와 저비용항공사(LCC) 간의 평균 지연 막대를 비교해 보면 편차가 크지 않고 유사하게 관리되고 있습니다. 다만 특정 LCC 항공사의 운항 지연 막대가 소폭 높게 서 있는 것은, 기재 부족이나 연속 회항 시스템 구조상 연계 지연(Connecting Delay)에 좀 더 취약하게 방치되어 있음을 가리킵니다.

---

## Step 4: 다변수 상관관계 및 이상치 분석 (Multivariate EDA)

![Step 4 상관관계 분석 개념도](img/step4_multivariate.svg)

두 개 이상의 변수를 동시에 결합하여, 조건에 따른 수치 차이나 독립 변수와 종속 변수 간의 통계적 경향을 분석합니다.


In [ ]:
plt.figure(figsize=(9, 6))

# 예정 시각(X) 대비 지연 분(Y)을 기상 악화(hue)에 결합한 산점도
sns.scatterplot(data=df, x='ScheduledTime_Hour', y='Delay_Minutes', hue='Weather_Severity', palette='Oranges', alpha=0.8)

plt.title('운항 예정 시간 및 기상 악화도별 지연 시간 분석', fontsize=14, fontweight='bold')
plt.xlabel('운항 예정 시간 (시)')
plt.ylabel('지연 시간 (분)')
plt.show()


> **💻 [실행 결과 시각화]**
> ![실행 결과 시각화](img/exec_step_4.svg)

### 💡 코드 딥다이브 & 비즈니스 통찰 (Analyst's Insight)
* **오후 누적 대기 체증 및 기상 시너지 규명:** 산점도를 보면 오전 시간대(6~11시)에는 기상 악화도가 다소 높더라도 지연 시간이 비교적 낮은 하단 대역에 묶여 있습니다. 하지만 오후 17시를 넘어서 야간으로 갈수록 짙은 주황색(심각한 기상 악화) 점들이 Y축 최상단(200분 이상)에 빈번하게 뭉쳐 있습니다. 이는 공항 관제망이 오전의 지연을 소화하지 못한 채 뒤편 항공편으로 **연계 지연을 누적 전달**하며 발생하는 공항 운영 병목 현상을 시각적으로 정교하게 대변해 줍니다.

---

## Step 5: 통계적 직관과 해석 (Statistical Logic)

> 💡 **[대기행렬 이론(Queueing Theory)과 관제 병목의 시계열성]**
> 공항이나 대규모 배송 허브 물류를 다룰 때는 독립 변수들이 서로 어떻게 시간에 종속되며 병목을 가중하는지 **대기행렬 이론(Queueing Theory)**으로 이해해야 합니다.
> * 항공기 1대가 활주로에 늦게 안착하면 뒤따르는 비행편들의 예정 슬롯이 전부 밀리는 현상이 발생하는데, 이러한 '시간 종속적 누적 성향'이 바로 산점도의 후반 지연 폭증 띠의 정체입니다.
> * 데이터 분석가가 교통 및 물류를 설계할 때 단순 일일 평균 지연 시간 지표만 보고하면 운영진은 핵심 병목 요인을 찾지 못합니다.
> * 시계열 예정 시간별로 분할하여 피크 체증의 가중 기여 기울기를 따로 집계하고, 병목 임계 시점(예: 오후 3시 이후 기재 연결 재배치)을 포착해 경보 체계를 탑재하는 것이 비즈니스 물류 최적화의 열쇠입니다.

---

## 🎯 30분 강의 마무리 및 심화 과제

오늘 우리는 실전 데이터셋을 분석하여 판다스로 데이터를 가공 및 정제하고, 시각화를 활용하여 핵심 변수 간의 통계적 유의성을 검증했습니다. 데이터 속에서 숨겨진 패턴을 올바른 시각으로 탐색하는 능력이 데이터 사이언티스트의 가장 강력한 무기입니다.

### 📝 심화 과제 (Advanced Challenge)
1. **주말(IsWeekend) 여부에 따른 평균 지연 격차 분석:** 평일과 주말의 평균 `Delay_Minutes`를 각각 구하고, 주말 여객 수요 증가가 관제 지연에 미치는 영향을 비교 집계하는 판다스 코드를 작성해 보세요.
2. **기상 악화 경보 기준 다중 필터링:** `Weather_Severity >= 4.0`인 재난급 기상 조건에서 전체 항공사들이 겪은 평균 지연 시간을 산출하고, 이 중 가장 정시성이 뛰어났던(지연 시간이 가장 적은) 탑 1개 항공사를 선별해 보세요.
